In [1]:
import json
import re
import os
from PIL import Image
import pdfplumber
import torch
import cv2
import numpy as np
# 
from transformers import pipeline
from output_utils import save_split_output


from confidence_utils import (
    calculate_eob_confidence,
    _unwrap_vlm_output,calculate_model_confidence
)
# =========================================================
# LOAD MODEL
# =========================================================

pipe = pipeline(
    "image-text-to-text",
    model="Qwen/Qwen3-VL-2B-Instruct",
    device_map="auto"
)

# =========================
# MODEL
# =========================
# from pipe_fn import pipe

# pipe = pipeline(
#     "image-text-to-text",
#     model="Qwen/Qwen3-VL-2B-Instruct",
#     device_map="auto"
# )


# =========================
# CROP TABLES
# =========================
def crop_claim_tables(pdf_path, output_dir="Personify_results"):
    os.makedirs(output_dir, exist_ok=True)
    cropped_images = []

    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages, start=1):
            print(f"\n📄 Processing Page {page_num}")

            claim_hits = page.search("Claim#:")
            if not claim_hits:
                claim_hits = page.search("Claim#")

            total_hits = page.search("Total Payment")

            if not claim_hits or not total_hits:
                continue

            table_count = min(len(claim_hits), len(total_hits))

            for idx in range(table_count):
                start_y = claim_hits[idx]["top"] - 5
                end_y   = total_hits[idx]["bottom"] + 10

                if start_y >= end_y:
                    continue

                bbox = (0, start_y, page.width, end_y)
                cropped_page = page.crop(bbox)

                image_path = os.path.join(
                    output_dir,
                    f"page_{page_num}_table_{idx+1}.png"
                )
                cropped_page.to_image(resolution=500).save(image_path)
                print(f"✅ Saved: {image_path}")

                expected_rows = count_service_rows(page, start_y, end_y)

                # Try exact text-layer extraction first (digital PDFs).
                # None means it couldn't locate a reliable header — the
                # pipeline will fall back to image+VLM for this table.
                text_extraction = extract_table_via_text(page, start_y, end_y)

                cropped_images.append({
                    "page":            page_num,
                    "table":           idx + 1,
                    "image_path":      image_path,
                    "expected_rows":   expected_rows,
                    "text_extraction": text_extraction,
                })

    return cropped_images


def count_service_rows(page, region_top, region_bottom):
    """
    Counts the number of distinct procedure-code rows (e.g. D0220, D0140)
    in the region. This is the count of Row A's (merged service line
    items) expected in the FINAL merged output — not the raw physical
    row count (which may include continuation rows with no code).
    """
    words = page.extract_words()
    service_rows = set()
    for w in words:
        text = w["text"].strip()
        if re.fullmatch(r"D\d{4}", text):
            y = float(w["top"])
            if region_top <= y <= region_bottom:
                service_rows.add(round(y, 1))
    return len(service_rows)


# =========================
# TEXT-BASED EXTRACTION (pdfplumber, no VLM / no image)
# =========================
# For a DIGITAL (non-scanned) PDF, the exact row text is already sitting
# in the PDF's text layer. Reading it directly via pdfplumber is 100%
# exact — no risk of a small VLM "giving up" and reporting a continuation
# row as blank when it actually shows $0.00 / $117.39 etc. We only fall
# back to the image+VLM path when this text-based approach can't locate
# a reliable header (e.g. a genuinely scanned page with no text layer).

AMOUNT_FIELDS = [
    "total_charges", "patient_responsibility", "copay_amount",
    "deductible_amount", "covered_expense", "payment_amount",
]

# left-to-right anchor word for each column, as literally printed in the
# header (first line of the two-line wrapped header)
_COLUMN_ANCHORS = {
    "date_of_service":        "Dates",
    "procedure_code":         "Procedure",
    "total_charges":          "Total",
    "patient_responsibility": "Excluded",
    "copay_amount":           "Co-pay",
    "deductible_amount":      "Deductible",
    "covered_expense":        "Covered",
    "paid_at":                "Paid",       # kept only to bound neighboring columns, discarded after
    "payment_amount":         "Payment",
}


def find_column_bounds(page, region_top, region_bottom):
    """
    Locates the table header row within the region and returns
    [(field_name, x_left, x_right), ...] sorted left to right. Returns
    None if the header can't be reliably located (caller should then
    fall back to the image+VLM path for that table).
    """
    words = page.extract_words()
    region_words = [w for w in words if region_top <= w["top"] <= region_bottom]

    provider_hits = [w for w in region_words if w["text"] == "Provider"]
    if not provider_hits:
        return None
    header_top = provider_hits[0]["top"]

    # header wraps across two stacked lines (e.g. "Dates of" / "Service")
    header_band = [w for w in region_words if header_top - 2 <= w["top"] <= header_top + 15]

    col_x0 = {}
    for field, anchor_text in _COLUMN_ANCHORS.items():
        hits = [w for w in header_band if w["text"] == anchor_text]
        if hits:
            col_x0[field] = hits[0]["x0"]

    # need every anchor found reliably, or we can't trust the mapping
    if len(col_x0) < len(_COLUMN_ANCHORS):
        return None

    ordered = sorted(col_x0.items(), key=lambda kv: kv[1])
    bounds = []
    for i, (field, x0) in enumerate(ordered):
        left = x0 - 5
        right = ordered[i + 1][1] - 2 if i + 1 < len(ordered) else page.width
        bounds.append((field, left, right))
    return bounds


def cluster_rows(words, tolerance=3):
    """Groups words into physical table rows based on similar 'top' y-values."""
    sorted_words = sorted(words, key=lambda w: (w["top"], w["x0"]))
    rows = []
    current_row = []
    current_top = None

    for w in sorted_words:
        if current_top is None or abs(w["top"] - current_top) <= tolerance:
            current_row.append(w)
            if current_top is None:
                current_top = w["top"]
        else:
            rows.append(current_row)
            current_row = [w]
            current_top = w["top"]

    if current_row:
        rows.append(current_row)

    return rows


def assign_row_to_columns(row_words, column_bounds):
    """Maps a row's words to field values using x-position column bounds."""
    result = {field: "" for field, _, _ in column_bounds}
    for w in sorted(row_words, key=lambda w: w["x0"]):
        cx = (w["x0"] + w["x1"]) / 2
        for field, left, right in column_bounds:
            if left <= cx < right:
                result[field] = (result[field] + " " + w["text"]).strip()
                break
    return result


def _clean_amount_text(text: str) -> str:
    return text.replace("$", "").replace(",", "").strip()


def extract_patient_name(page, region_top, region_bottom):
    hits = page.search("Patient:")
    target = None
    for h in hits:
        if region_top <= h["top"] <= region_bottom:
            target = h
            break
    if not target:
        return ""

    line_top, line_bottom = target["top"] - 1, target["bottom"] + 1
    words = page.extract_words()
    line_words = sorted(
        [w for w in words if line_top <= w["top"] <= line_bottom],
        key=lambda w: w["x0"],
    )

    # avoid bleeding into a same-line "Patient#:" label, if present
    stop_x = None
    hash_hits = [w for w in line_words if w["text"].startswith("Patient#")]
    if hash_hits:
        stop_x = hash_hits[0]["x0"]

    name_words = []
    for w in line_words:
        if w["x0"] < target["x1"] - 2:
            continue
        if stop_x is not None and w["x0"] >= stop_x - 2:
            break
        name_words.append(w["text"])

    return " ".join(name_words).strip()


def extract_column_totals(page, region_top, region_bottom, column_bounds):
    hits = page.search("Column Totals")
    target = None
    for h in hits:
        if region_top <= h["top"] <= region_bottom:
            target = h
            break
    if not target:
        return {}

    row_top, row_bottom = target["top"] - 2, target["bottom"] + 2
    words = page.extract_words()
    row_words = [w for w in words if row_top <= w["top"] <= row_bottom]

    raw_totals = assign_row_to_columns(row_words, column_bounds)
    return {f: _clean_amount_text(raw_totals.get(f, "")) for f in AMOUNT_FIELDS}


def extract_table_via_text(page, region_top, region_bottom):
    """
    Attempts to build the full raw per-row service list, patient name,
    and column totals directly from the PDF's text layer. Returns None
    if the header/columns can't be reliably located (caller falls back
    to image+VLM). This is exact and never "reads a continuation row as
    blank" the way a small VLM can.
    """
    column_bounds = find_column_bounds(page, region_top, region_bottom)
    if not column_bounds:
        return None

    words = page.extract_words()
    region_words = [w for w in words if region_top <= w["top"] <= region_bottom]

    provider_hits = [w for w in region_words if w["text"] == "Provider"]
    if not provider_hits:
        return None
    header_top = provider_hits[0]["top"]
    header_band = [w for w in region_words if header_top - 2 <= w["top"] <= header_top + 15]
    header_bottom = max(w["bottom"] for w in header_band) if header_band else header_top + 15

    totals_hits = [h for h in page.search("Column Totals") if region_top <= h["top"] <= region_bottom]
    body_bottom = totals_hits[0]["top"] - 2 if totals_hits else region_bottom

    body_words = [w for w in region_words if header_bottom + 2 <= w["top"] <= body_bottom]
    if not body_words:
        return None

    # strip "Paid At %" tokens (e.g. "0%", "100%") entirely — never wanted
    body_words = [w for w in body_words if not re.fullmatch(r"\d{1,3}%", w["text"])]

    row_clusters = cluster_rows(body_words, tolerance=3)

    raw_services = []
    for row_words in row_clusters:
        fields = assign_row_to_columns(row_words, column_bounds)
        fields.pop("paid_at", None)
        for f in AMOUNT_FIELDS:
            fields[f] = _clean_amount_text(fields.get(f, ""))
        raw_services.append(fields)

    if not raw_services:
        return None

    return {
        "patient_name": extract_patient_name(page, region_top, region_bottom),
        "services": raw_services,
        "column_totals": extract_column_totals(page, region_top, region_bottom, column_bounds),
    }


# =========================
# TABLE ENHANCEMENT
# =========================
def make_table(image_path):
    img  = cv2.imread(image_path)
    gray = cv2.imread(image_path, 0)
    _, thresh = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY_INV)
    sums  = np.sum(thresh, axis=1)
    th    = (thresh.shape[1] * 255) * 0.6
    lines = np.where(sums > th)[0]
    for l in lines:
        cv2.line(img, (0, l), (thresh.shape[1], l), (0, 0, 0), 1)
    return img


def convert_amounts_to_string(obj):
    amount_fields = {
        "total_charges", "patient_responsibility", "copay_amount",
        "deductible_amount", "covered_expense", "payment_amount",
    }
    if isinstance(obj, dict):
        new_obj = {}
        for k, v in obj.items():
            if k in amount_fields:
                try:
                    new_obj[k] = f"{float(str(v).replace('$','').replace(',','').strip()):.2f}"
                except Exception:
                    new_obj[k] = ""
            else:
                new_obj[k] = convert_amounts_to_string(v)
        return new_obj
    elif isinstance(obj, list):
        return [convert_amounts_to_string(i) for i in obj]
    else:
        return obj


# =========================
# PROMPT — RAW PER-ROW EXTRACTION ONLY (no merging, no math)
# =========================
def build_prompt(pdf_name: str) -> str:
    return """
You are extracting data from a dental EOB table, ROW BY ROW.

DO NOT merge rows. DO NOT sum anything. DO NOT decide which row
belongs to which service. Just report each PHYSICAL row in the table
exactly as printed, top to bottom, as its own separate JSON object.

========================
PATIENT NAME
========================

Extract patient_name ONLY from the text immediately following the label
"Patient:" (character-by-character, exactly as printed).

NEVER extract patient name from: Insured, Subscriber, Provider,
Patient#, or any other location. The "Insured" name is a DIFFERENT
person from the Patient and must never be used or blended in.

Example:
Insured   David L Herman
Patient: David Herman

Output:
"patient_name": "David Herman"

========================
ROW-BY-ROW EXTRACTION (THE MAIN TASK)
========================

Look at the table body, one printed row at a time, top to bottom.
For EVERY physical row in the table (whether or not it has a Line No.,
Dates of Service, or Procedure Code), output ONE JSON object with these
fields, using ONLY what is literally printed in THAT row:

- date_of_service   -> value under "Dates of Service" column, or "" if blank in this row
- procedure_code    -> value under "Procedure Code" column, or "" if blank in this row
- total_charges     -> value under "Total Charges" column, or "" if blank in this row
- patient_responsibility -> value under "Excluded Charges" column, or "" if blank in this row
- copay_amount      -> value under "Co-pay Amount" column, or "" if blank in this row
- deductible_amount -> value under "Deductible Amount" column, or "" if blank in this row
- covered_expense   -> value under "Covered Expense" column, or "" if blank in this row
- payment_amount    -> value under "Payment Amount" column, or "" if blank in this row

RULES:
1. Extract ONLY from the exact column each value is printed in. NEVER
   shift values left or right. NEVER pull a value from a neighboring
   column.
2. If a cell is blank in a given row, output "" for that field in that
   row's object. Do NOT guess, do NOT borrow a value from another row,
   do NOT leave a field out of the JSON object.
3. A row with no Line No. / no Dates of Service / no Procedure Code is
   STILL its own row — output it as its own object with those fields
   as "" and only the amount fields it actually shows filled in.
4. Do NOT skip any row. Do NOT combine two printed rows into one
   object. Do NOT try to figure out which service a blank-code row
   "belongs to" — that is handled separately, not by you.
5. Ignore the "Paid At %" column entirely — never extract it.
6. Process rows in the exact top-to-bottom order they appear.
7. CRITICAL — a continuation row (no Line No./date/code) almost always
   STILL has visible dollar amounts printed in it (e.g. $0.00, $0.00,
   $0.00, $117.39). You MUST read and report every amount that is
   visibly printed in that row, column by column, exactly like you
   would for any other row. Leaving an amount field as "" is ONLY
   correct when that specific cell is genuinely empty/blank on the
   page — NOT simply because the row has no Line No. or procedure
   code. Never output a continuation row where every field is "" if
   the row visibly shows any dollar figures at all — that means you
   failed to read the row and must look again.

Example table 1 (continuation row has a mix of zero and non-zero values):

Line 001  08/04-08/04/2023  D6057  Total=1100.63  Excluded=560.66  Co-pay=0.00  Deductible=0.00  Covered=0.00   Paid At=0%   Payment=0.00
                                                    Excluded=101.89              Co-pay=0.00  Deductible=0.00  Covered=438.08 Paid At=50%  Payment=219.04
Line 002  08/04-08/04/2023  D6058  Total=1626.83  Excluded=1626.83 Co-pay=0.00  Deductible=0.00  Covered=0.00   Paid At=0%   Payment=0.00

Correct output for "services" (THREE objects, one per printed row):

[
  {
    "date_of_service": "08/04-08/04/2023",
    "procedure_code": "D6057",
    "total_charges": "1100.63",
    "patient_responsibility": "560.66",
    "copay_amount": "0.00",
    "deductible_amount": "0.00",
    "covered_expense": "0.00",
    "payment_amount": "0.00"
  },
  {
    "date_of_service": "",
    "procedure_code": "",
    "total_charges": "",
    "patient_responsibility": "101.89",
    "copay_amount": "0.00",
    "deductible_amount": "0.00",
    "covered_expense": "438.08",
    "payment_amount": "219.04"
  },
  {
    "date_of_service": "08/04-08/04/2023",
    "procedure_code": "D6058",
    "total_charges": "1626.83",
    "patient_responsibility": "1626.83",
    "copay_amount": "0.00",
    "deductible_amount": "0.00",
    "covered_expense": "0.00",
    "payment_amount": "0.00"
  }
]

Notice the middle object has no date/code/total — that is correct and
expected. Do not try to merge it with a neighboring row. Do not drop it.

Example table 2 (Row A is fully $0.00, continuation row carries the ACTUAL
payment — this is common when a procedure is paid at 100%. Read the
continuation row's numbers just as carefully as Row A's):

Line 001  06/13-06/13/2023  D1110  Total=117.39  Excluded=0.00  Co-pay=0.00  Deductible=0.00  Covered=0.00   Paid At=0%    Payment=0.00
                                                    Excluded=0.00              Co-pay=0.00  Deductible=0.00  Covered=117.39 Paid At=100%  Payment=117.39

Correct output for "services" (TWO objects — do NOT leave the second one
blank just because Row A above it was all zeros):

[
  {
    "date_of_service": "06/13-06/13/2023",
    "procedure_code": "D1110",
    "total_charges": "117.39",
    "patient_responsibility": "0.00",
    "copay_amount": "0.00",
    "deductible_amount": "0.00",
    "covered_expense": "0.00",
    "payment_amount": "0.00"
  },
  {
    "date_of_service": "",
    "procedure_code": "",
    "total_charges": "",
    "patient_responsibility": "0.00",
    "copay_amount": "0.00",
    "deductible_amount": "0.00",
    "covered_expense": "117.39",
    "payment_amount": "117.39"
  }
]

WRONG output for that same row (do NOT do this — the row is not empty,
it clearly shows $0.00, $0.00, $0.00, $117.39, $117.39; reporting it as
all "" means you did not actually read the row):

  {
    "date_of_service": "",
    "procedure_code": "",
    "total_charges": "",
    "patient_responsibility": "",
    "copay_amount": "",
    "deductible_amount": "",
    "covered_expense": "",
    "payment_amount": ""
  }

========================
TOTALS EXTRACTION
========================

Extract totals ONLY from the row labeled "Column Totals". Do NOT
calculate totals yourself. Do NOT use any service rows for totals.

Extract ONLY these totals:
total_charges, total_responsibility, copay_amount, deductible_amount,
covered_expense, payment_amount
========================
OUTPUT FORMAT
========================

Return ONLY valid JSON, nothing else.

IMPORTANT:
EVERY extracted field MUST contain both:
- "value"
- "confidence"

NEVER return any extracted field as a plain string.

Confidence rules:
- confidence MUST be a number between 0.0 and 1.0.
- 1.0 = completely certain that the value was correctly read.
- 0.0 = value is missing, unreadable, or cannot be reliably extracted.
- Do NOT guess values.
- If a value cannot be reliably extracted, return:
  {
    "value": "",
    "confidence": 0.0
  }
- Confidence represents ONLY confidence in reading the value from the image.
- Do NOT calculate confidence based on the financial amount.
- Do NOT calculate confidence based on validation results.

The JSON MUST follow this exact structure:

{
  "patient_name": {
    "value": "",
    "confidence": 0.0
  },

  "services": [
    {
      "date_of_service": {
        "value": "",
        "confidence": 0.0
      },

      "procedure_code": {
        "value": "",
        "confidence": 0.0
      },

      "total_charges": {
        "value": "",
        "confidence": 0.0
      },

      "patient_responsibility": {
        "value": "",
        "confidence": 0.0
      },

      "copay_amount": {
        "value": "",
        "confidence": 0.0
      },

      "deductible_amount": {
        "value": "",
        "confidence": 0.0
      },

      "covered_expense": {
        "value": "",
        "confidence": 0.0
      },

      "payment_amount": {
        "value": "",
        "confidence": 0.0
      }
    }
  ],

  "column_totals": {
    "total_charges": {
      "value": "",
      "confidence": 0.0
    },

    "patient_responsibility": {
      "value": "",
      "confidence": 0.0
    },

    "copay_amount": {
      "value": "",
      "confidence": 0.0
    },

    "deductible_amount": {
      "value": "",
      "confidence": 0.0
    },

    "covered_expense": {
      "value": "",
      "confidence": 0.0
    },

    "payment_amount": {
      "value": "",
      "confidence": 0.0
    }
  }
}
 For every extracted field, return:
   - value
   - confidence
 
VALUE + CONFIDENCE RULES:
 
For every field return:
{
  "value": "",
  "confidence": ""
}
 
VALUE:
- "value" = the exact text/value visibly present in the specified location.
- Read ONLY from the exact cell/row/column requested.
- Copy exactly as printed; preserve "$" and formatting when visible.
- Never guess, infer, calculate, copy, shift, or use values from another row,
  column, table section, or Totals row.
- If the exact location is blank, missing, or has no clearly readable value:
  value = ""
 
CONFIDENCE:
- "confidence" = confidence that the extracted value is actually present
  in that exact location.
- Use a number from 0.0 to 1.0 based ONLY on visual evidence.
- 1.0 = clearly visible and certain.
- 0.8–0.99 = clearly visible with minor uncertainty.
- 0.5–0.79 = visible but difficult/ambiguous.
- 0.1–0.49 = very unclear.
- 0.0 = blank, missing, or no reliable visual evidence.
 
IMPORTANT:
Confidence is NOT confidence that the value is mathematically correct
or logically expected. It is ONLY confidence that the value shown in
"value" is what is visibly printed in the exact requested location.
 
If value = "":
confidence MUST = 0.0.
"""

def extract_payor_name(pdf_path, top_region_height=180, x_tolerance=15):
    """
    Extracts the payor company name from the top-left letterhead block
    of the FIRST page. Logic: find the line containing "PO BOX", then
    take the line immediately above it that shares (roughly) the same
    left x-position — that's the address block's company name line,
    not the big stylized logo above it.

    Returns None if it can't confidently locate it.
    """
    with pdfplumber.open(pdf_path) as pdf:
        page = pdf.pages[0]
        words = page.extract_words()

        top_words = [w for w in words if w["top"] < top_region_height]
        if not top_words:
            return None

        rows = cluster_rows(top_words, tolerance=3)
        rows = sorted(rows, key=lambda r: min(w["top"] for w in r))

        po_box_idx = None
        po_box_x0 = None
        for i, row in enumerate(rows):
            line_text = " ".join(w["text"] for w in row).upper()
            if "PO BOX" in line_text or "P.O. BOX" in line_text or "P O BOX" in line_text:
                po_box_idx = i
                po_box_x0 = min(w["x0"] for w in row)
                break

        if po_box_idx is None or po_box_idx == 0:
            return None

        # walk upward from the line just above "PO BOX" until we find
        # a line whose left edge lines up with the PO BOX line
        for i in range(po_box_idx - 1, -1, -1):
            row = rows[i]
            row_x0 = min(w["x0"] for w in row)
            if abs(row_x0 - po_box_x0) <= x_tolerance:
                payor_words = sorted(row, key=lambda w: w["x0"])
                payor_name = " ".join(w["text"] for w in payor_words).strip()
                return payor_name
            # if it doesn't line up, keep looking one line further up
            # (in case there's a blank/logo line in between with a
            # different x0), but stop after a couple of tries so we
            # don't wander into unrelated header text
            if po_box_idx - i >= 3:
                break

        return None

# =========================
# AMOUNT HELPERS
# =========================
def parse_amount(x) -> float:
    if x is None or str(x).strip() == "":
        return 0.0
    try:
        return float(str(x).replace("$", "").replace(",", "").strip())
    except ValueError:
        return 0.0


def compute_totals_from_services(services: list) -> dict:
    fields = [
        "total_charges",
        "patient_responsibility",
        "copay_amount",
        "deductible_amount",
        "covered_expense",
        "payment_amount",
    ]
    return {
        f: round(sum(parse_amount(s.get(f, "")) for s in services), 2)
        for f in fields
    }


def services_are_empty(services: list) -> bool:
    """Return True when every amount field in every service row is blank/zero."""
    amount_fields = [
        "total_charges",
        "patient_responsibility",
        "copay_amount",
        "deductible_amount",
        "covered_expense",
        "payment_amount",
    ]
    for svc in services:
        for f in amount_fields:
            if str(svc.get(f, "")).strip() not in ("", "0.00", "0"):
                return False
    return True


# =========================
# MERGE — deterministic Row A / Row B combination (moved out of the VLM)
# =========================
def merge_continuation_rows(raw_services: list) -> list:
    """
    Takes the RAW per-physical-row extraction (one object per printed
    row, some with no date/procedure_code) and merges continuation rows
    (no date AND no procedure_code) UPWARD into the nearest preceding
    row that has a date or procedure_code.

    - date_of_service / procedure_code / total_charges: taken from Row A only.
    - patient_responsibility / copay_amount / deductible_amount /
      covered_expense / payment_amount: SUMMED across Row A + any Row B
      rows that follow it, until the next Row A starts.
    """
    merged = []

    for row in raw_services:
        has_code = str(row.get("procedure_code", "")).strip() != ""
        has_date = str(row.get("date_of_service", "")).strip() != ""

        if has_code or has_date:
            # Starts a new service record (Row A)
            merged.append({
                "date_of_service": row.get("date_of_service", ""),
                "procedure_code": row.get("procedure_code", ""),
                "total_charges": row.get("total_charges", ""),
                "patient_responsibility": parse_amount(row.get("patient_responsibility", "")),
                "copay_amount": parse_amount(row.get("copay_amount", "")),
                "deductible_amount": parse_amount(row.get("deductible_amount", "")),
                "covered_expense": parse_amount(row.get("covered_expense", "")),
                "payment_amount": parse_amount(row.get("payment_amount", "")),
            })
        else:
            # Continuation row (Row B) — belongs to the last Row A seen
            if not merged:
                print(f"  ⚠️  Orphan continuation row with no preceding Row A, skipping: {row}")
                continue
            last = merged[-1]
            last["patient_responsibility"] += parse_amount(row.get("patient_responsibility", ""))
            last["copay_amount"] += parse_amount(row.get("copay_amount", ""))
            last["deductible_amount"] += parse_amount(row.get("deductible_amount", ""))
            last["covered_expense"] += parse_amount(row.get("covered_expense", ""))
            last["payment_amount"] += parse_amount(row.get("payment_amount", ""))

    # Format amount fields back to 2-decimal strings
    amount_fields = [
        "patient_responsibility", "copay_amount",
        "deductible_amount", "covered_expense", "payment_amount",
    ]
    for m in merged:
        for f in amount_fields:
            m[f] = f"{m[f]:.2f}"

    return merged


# =========================
# JSON CLEANER
# =========================
def extract_json(text: str) -> dict:
    start = text.find("{")
    end   = text.rfind("}") + 1
    if start == -1 or end == 0:
        raise ValueError("No JSON object found in model output")
    return json.loads(text[start:end])


def save_json(data, output_path: str):
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)


# =========================
# VALIDATION — RAW (unmerged) output
# =========================
# Field totals (total_charges, patient_responsibility, etc.) are just
# column sums, and summing over ALL raw rows (Row A + Row B together)
# gives the exact same result merging would — Row B never duplicates a
# Row A value, it only adds its own portion. So the field-by-field
# totals check is safe to run directly on raw/unmerged services; we
# just also check row count is at least the number of procedure codes
# expected (raw rows can include extra continuation rows, never fewer).
def validate_raw_rows(patient: dict, patient_name: str, expected_min_rows: int):
    services = patient.get("services", [])
    totals   = patient.get("totals", {})

    field_names = list(compute_totals_from_services([]).keys())   # ADD
    total_fields = len(field_names)      

    if not services:
        field_errors = [                                            # ADD
            {"field": f, "computed": 0.0, "extracted": None}
            for f in field_names
        ]
        return False, "No services found", [{"error": "empty services"}] + field_errors, total_fields

    if services_are_empty(services):
        msg = "All service rows are empty — model likely failed to extract data"
        print(f"\n❌ {msg}")
        field_errors = [                                            # ADD
            {"field": f, "computed": 0.0, "extracted": None}
            for f in field_names
        ]
        return False, msg, [{"error": "all_service_rows_empty"}] + field_errors, total_fields

    computed_totals = compute_totals_from_services(services)
    result_validation = ""
    errors = []
    has_error = False

    print(f"\n🔍 Validation for [{patient_name}]")
    print("-" * 80)

    for field, computed_value in computed_totals.items():
        extracted_value = round(parse_amount(totals.get(field, "")), 2)
        diff  = round(computed_value - extracted_value, 2)
        match = abs(diff) <= 0.01
        icon   = "✅" if match else "❌"
        status = "MATCH" if match else "MISMATCH"

        if not match:
            has_error = True
            errors.append({
                "type":       "field_mismatch",
                "field":      field,
                "computed":   computed_value,
                "extracted":  extracted_value,
                "difference": diff,
            })

        line = (
            f"{icon} {field:25s} computed={computed_value:<10} "
            f"| extracted={extracted_value:<10} {status}"
        )
        print(line)
        result_validation += "\n" + line

    # Count only "Row A" entries — rows that actually represent a service
    # (have a procedure_code or date_of_service) — NOT raw physical rows.
    # expected_min_rows is a procedure-code count, so this needs to be
    # counted the same way to mean the same thing.
    extracted_row_count = sum(
        1 for s in services
        if str(s.get("procedure_code", "")).strip() != ""
        or str(s.get("date_of_service", "")).strip() != ""
    )
    match  = extracted_row_count >= expected_min_rows
    icon   = "✅" if match else "❌"
    status = "MATCH" if match else "TOO FEW ROWS"

    if not match:
        has_error = True
        errors.append({
            "type":              "raw_row_count_too_low",
            "expected_min_rows": expected_min_rows,
            "extracted_rows":    extracted_row_count,
        })

    line = (
        f"{icon} {'total_record_rows':25s} computed={expected_min_rows:<10} "
        f"| extracted={extracted_row_count:<10} {status}"
    )
    print(line)
    result_validation += "\n" + line
    print("-" * 80)

    if has_error:
        print(f"❌ [{patient_name}] Validation FAILED\n")
        return False, result_validation, errors, total_fields
    else:
        print(f"✅ [{patient_name}] Validation PASSED\n")
        return True, result_validation, [], total_fields


# =========================
# DENIAL CHECK
# =========================
def check_claim_denied(pdf_path: str) -> str:
    denial_keywords = ["denied", "denial"]
    stop_phrase     = "Appealing a denial of the dental claim"

    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            full_text = page.extract_text()
            if not full_text:
                continue
            searchable = full_text.lower()
            if stop_phrase.lower() in searchable:
                searchable = searchable.split(stop_phrase.lower())[0]
            for keyword in denial_keywords:
                if keyword in searchable:
                    print(f"claim denied keyword found: {keyword}")
                    return "denied"

    return "not denied"


# =========================
# RETRY HELPER — extracts RAW rows, merges, validates the MERGED result
# =========================
MAX_RETRIES = 2

def run_model_with_retry(image: Image.Image, prompt: str, expected_rows: int):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text",  "text":  prompt},
            ],
        }
    ]

    last_parsed = None
    last_raw_services = None

    for attempt in range(1, MAX_RETRIES + 1):
        print(f"  🔄 Model attempt {attempt}/{MAX_RETRIES}")

        with torch.no_grad():
            output = pipe(
                messages,
                max_new_tokens=1500,
                temperature=0.0,
                do_sample=False,
            )

        raw = output[0]["generated_text"]
        if isinstance(raw, list):
            raw = raw[-1]["content"]

        try:
            parsed = extract_json(raw)
        except Exception as e:
            print(f"  ❌ JSON parse failed on attempt {attempt}: {e}")
            continue

        raw_services = parsed.get("services", [])

        # NOTE: no merging here anymore — return the raw per-physical-row
        # extraction as-is. Row A/Row B merging is done separately, later,
        # by calling merge_continuation_rows() explicitly when you're ready.

        # Raw rows can never be fewer than the number of procedure codes
        # expected on the page (each procedure code IS a physical row);
        # they can be more, since continuation rows add extra rows.
        row_ok   = (len(raw_services) >= expected_rows)
        empty_ok = not services_are_empty(raw_services)

        if row_ok and empty_ok:
            print(f"  ✅ Accepted on attempt {attempt} (raw rows={len(raw_services)}, expected >= {expected_rows})")
            return parsed

        print(
            f"  ⚠️  attempt {attempt}: raw_rows={len(raw_services)} "
            f"(expected >= {expected_rows}), all_empty={not empty_ok}"
        )
        last_parsed = parsed
        last_raw_services = raw_services

    print(f"  ⚠️  All {MAX_RETRIES} attempts exhausted — using last result")
    return last_parsed

#date_formate_change
def normalize_date_of_service(obj):
    """
    Converts:
        10/21-10/21/2022  -> 10/21/2022
        08/04-08/04/2023  -> 08/04/2023

    Works recursively for dicts/lists.
    """

    pattern = re.compile(r"^(\d{2}/\d{2})-\d{2}/\d{2}/(\d{4})$")

    if isinstance(obj, dict):
        for key, value in obj.items():
            if key == "date_of_service" and isinstance(value, str):
                m = pattern.match(value.strip())
                if m:
                    obj[key] = f"{m.group(1)}/{m.group(2)}"
            else:
                normalize_date_of_service(value)

    elif isinstance(obj, list):
        for item in obj:
            normalize_date_of_service(item)

    return obj

# =========================
# MAIN PIPELINE
# =========================
def run_pipeline(pdf_path: str, output_dir: str = "EOB_OUTPUT/Personify_health", company_name= "Personify Health"):
    # pdf_name is used as eob_id throughout
    pdf_full_name = os.path.basename(pdf_path)
    pdf_name = os.path.basename(pdf_path).split(".")[0].split("_")[-1]

    base_dir    = os.path.join(output_dir, pdf_name)
    cropped_dir = os.path.join(base_dir, "cropped_images")
    json_output_path = os.path.join(base_dir, f"{pdf_name}_output.json")

    if os.path.exists(json_output_path):
        print(f"⏭️  Skipping {pdf_name} — output already exists")
        return None

    os.makedirs(base_dir,    exist_ok=True)
    os.makedirs(cropped_dir, exist_ok=True)

    image_paths  = crop_claim_tables(pdf_path, output_dir=cropped_dir)
    is_denied    = check_claim_denied(pdf_path)
    print(f"claim status: {is_denied}")

    # Build prompt once with pdf_name as eob_id
    final_prompt = build_prompt(pdf_name)

    results = []

    payor_name = extract_payor_name(pdf_path) or "UNKNOWN PAYOR" 

    for idx, item in enumerate(image_paths):
        img_path         = item["image_path"]
        expected_rows    = item["expected_rows"]
        text_extraction  = item.get("text_extraction")

        print(f"\nProcessing table {idx+1}/{len(image_paths)}")

        # Prefer the exact text-layer extraction — it's not subject to the
        # VLM misreading a continuation row as blank. Only fall back to
        # the image+VLM path if text extraction wasn't possible (e.g. a
        # scanned page with no text layer) or looks structurally broken.
        use_text = (
            text_extraction is not None
            and len(text_extraction.get("services", [])) >= expected_rows
            and not services_are_empty(text_extraction.get("services", []))
        )

        if use_text:
            print(f"  ✅ Using exact text-layer extraction (no VLM needed)")
            parsed = {
                "patient_name":  text_extraction["patient_name"],
                "services":      text_extraction["services"],
                "column_totals": text_extraction["column_totals"],
                "_model_confidence": 1.0,
            }
        else:
            print(f"  ⚠️  Text-layer extraction unavailable/unreliable — falling back to image+VLM")
            image_cv  = make_table(img_path)
            image_pil = Image.fromarray(image_cv).convert("RGB")
            parsed = run_model_with_retry(image_pil, final_prompt, expected_rows)

            # Only the VLM path returns {"value":..., "confidence":...} wrappers —
            # only it needs unwrapping / confidence calculation.
            if parsed is not None:
                model_confidence = calculate_model_confidence(parsed)
                parsed = _unwrap_vlm_output(parsed)
                parsed["_model_confidence"] = model_confidence        # ADD

        parsed["eob_id"] = pdf_name
        parsed = convert_amounts_to_string(parsed)
        parsed["_expected_rows"] = expected_rows

        print("Extracted (merged):")
        print(json.dumps(parsed, indent=2))

        date_of_service = ""
        if parsed.get("services"):
            date_of_service = parsed["services"][0].get("date_of_service", "")

        patient_data = {
            "patient_name": parsed.get("patient_name", ""),
            "date_of_service": date_of_service,
            "services": parsed.get("services", []),
            "totals": parsed.get("column_totals", {}),
            "_model_confidence": parsed.get("_model_confidence", 0.0),
        }

        is_valid, log, errors, total_fields= validate_raw_rows(
            patient=patient_data,
            patient_name=patient_data.get("patient_name", "UNKNOWN"),
            expected_min_rows=expected_rows,
        )

        patient_data["validation"] = {
            "status": is_valid,
            "errors": errors,
        }
        patient_data["_total_fields"] = total_fields 

        # Validation runs on the RAW (unmerged) rows above — sums are the
        # same either way, so this doesn't change any validation result.
        # But the FINAL saved JSON should show the merged view: each
        # continuation row's amounts folded upward into its Row A, so
        # "services" ends up with exactly one object per procedure code.
        patient_data["services"] = merge_continuation_rows(patient_data["services"])
        normalize_date_of_service(patient_data)
        results.append(patient_data)
        payor_name = extract_payor_name(pdf_path)
        if not payor_name:
            payor_name = "UNKNOWN PAYOR"   # fallback so pipeline never crashes

    confidence_results = list(results)                              # ADD — insert here
    confidence_score = calculate_eob_confidence(confidence_results)  # ADD — insert here

    for patient in results:                                          # ADD — insert here
        patient.pop("_total_fields", None)
        patient.pop("_model_confidence", None)

    final = [
        {
            "eob_id":       pdf_name,
            "file_name":pdf_full_name,
            "claim_status": is_denied,
            "payor": payor_name,
            "confidence_score": confidence_score,
            "patients":     results,
        }
    ]

    success_path, failed_path = save_split_output(
                                        final,
                                        company_name=company_name,
                                        pdf_name=pdf_name,
                                        pdf_path=pdf_path,
                                        cropped_dir=cropped_dir,
                                    )
                                
    print(f"\n📁 Cropped images : {cropped_dir}")
    print(f"✅ Success json   : {success_path}")
    print(f"⚠  Failed json    : {failed_path}")
    return final

W0901 19:19:45.352000 3530569 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0901 19:19:45.367000 3530569 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

In [2]:
run_pipeline(r"/home/cipl/users/OCR_Project/Solution_15_04/Jeeva/Personify_health/pdfs/Pmt_EOP_336494100.pdf")


📄 Processing Page 1
✅ Saved: EOB_OUTPUT/Personify_health/336494100/cropped_images/page_1_table_1.png

📄 Processing Page 2
✅ Saved: EOB_OUTPUT/Personify_health/336494100/cropped_images/page_2_table_1.png
claim status: not denied

Processing table 1/2
  ✅ Using exact text-layer extraction (no VLM needed)
Extracted (merged):
{
  "patient_name": "David Herman",
  "services": [
    {
      "date_of_service": "03/02-03/02/2023",
      "procedure_code": "D6010",
      "total_charges": "2245.88",
      "patient_responsibility": "0.00",
      "copay_amount": "0.00",
      "deductible_amount": "50.00",
      "covered_expense": "2245.88",
      "payment_amount": "1097.94"
    }
  ],
  "column_totals": {
    "total_charges": "2245.88",
    "patient_responsibility": "0.00",
    "copay_amount": "0.00",
    "deductible_amount": "50.00",
    "covered_expense": "2245.88",
    "payment_amount": "1097.94"
  },
  "_model_confidence": 1.0,
  "eob_id": "336494100",
  "_expected_rows": 1
}

🔍 Validation for

[{'eob_id': '336494100',
  'file_name': 'Pmt_EOP_336494100.pdf',
  'claim_status': 'not denied',
  'payor': 'HealthComp Explanation of Benefits',
  'confidence_score': 100.0,
  'patients': [{'patient_name': 'David Herman',
    'date_of_service': '03/02/2023',
    'services': [{'date_of_service': '03/02/2023',
      'procedure_code': 'D6010',
      'total_charges': '2245.88',
      'patient_responsibility': '0.00',
      'copay_amount': '0.00',
      'deductible_amount': '50.00',
      'covered_expense': '2245.88',
      'payment_amount': '1097.94'}],
    'totals': {'total_charges': '2245.88',
     'patient_responsibility': '0.00',
     'copay_amount': '0.00',
     'deductible_amount': '50.00',
     'covered_expense': '2245.88',
     'payment_amount': '1097.94'},
    'validation': {'status': True, 'errors': []}},
   {'patient_name': 'David Herman',
    'date_of_service': '03/16/2023',
    'services': [{'date_of_service': '03/16/2023',
      'procedure_code': 'D0120',
      'total_charge

In [ ]:
run_pipeline(r"/home/cipl/users/OCR_Project/Solution_15_04/Jeeva/Personify_health/pdfs/Pmt_EOP_280247484.pdf")


📄 Processing Page 1
✅ Saved: EOB_OUTPUT/Personify_health/280247484/cropped_images/page_1_table_1.png
claim status: not denied

Processing table 1/1
  ✅ Using exact text-layer extraction (no VLM needed)
Extracted (merged):
{
  "patient_name": "David Herman",
  "services": [
    {
      "date_of_service": "09/15-09/15/2022",
      "procedure_code": "D0120",
      "total_charges": "65.63",
      "patient_responsibility": "0.00",
      "copay_amount": "0.00",
      "deductible_amount": "0.00",
      "covered_expense": "65.63",
      "payment_amount": "65.63"
    },
    {
      "date_of_service": "09/15-09/15/2022",
      "procedure_code": "D1110",
      "total_charges": "117.39",
      "patient_responsibility": "0.00",
      "copay_amount": "0.00",
      "deductible_amount": "0.00",
      "covered_expense": "117.39",
      "payment_amount": "117.39"
    },
    {
      "date_of_service": "09/15-09/15/2022",
      "procedure_code": "D0274",
      "total_charges": "84.57",
      "patient_res

[{'eob_id': '280247484',
  'claim_status': 'not denied',
  'payor': 'GILSBAR, L.L.C. Explanation of Benefits',
  'confidence_score': 100.0,
  'patients': [{'patient_name': 'David Herman',
    'date_of_service': '09/15/2022',
    'services': [{'date_of_service': '09/15/2022',
      'procedure_code': 'D0120',
      'total_charges': '65.63',
      'patient_responsibility': '0.00',
      'copay_amount': '0.00',
      'deductible_amount': '0.00',
      'covered_expense': '65.63',
      'payment_amount': '65.63'},
     {'date_of_service': '09/15/2022',
      'procedure_code': 'D1110',
      'total_charges': '117.39',
      'patient_responsibility': '0.00',
      'copay_amount': '0.00',
      'deductible_amount': '0.00',
      'covered_expense': '117.39',
      'payment_amount': '117.39'},
     {'date_of_service': '09/15/2022',
      'procedure_code': 'D0274',
      'total_charges': '84.57',
      'patient_responsibility': '0.00',
      'copay_amount': '0.00',
      'deductible_amount': '0.00

: 